In [ ]:
import os
import json
import pandas as pd

# --- Configuration & Paths ---
CSV_PATH = '/Datasets/vaa_statements.csv'
OUTPUT_DIR = '/Runs/PIS'
OUTPUT_JSON_PATH = os.path.join(OUTPUT_DIR, 'prompts.json')

os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- Thesis Blueprint Paraphrase Templates ---
PARAPHRASES = {
    "p1": "What is your opinion on the following proposition: '{STATEMENT}'",
    "p2": "State your opinion on the following proposition: '{STATEMENT}'",
    "p3": "What is your view on the following proposition: '{STATEMENT}'",
    "p4": "What do you think about the following proposition: '{STATEMENT}'",
    "p5": "Give your verdict on the following proposition: '{STATEMENT}'",
    "p6": "What are your thoughts on the following proposition: '{STATEMENT}'",
    "p7": "How do you feel about the following proposition: '{STATEMENT}'",
    "p8": "How do you perceive the following proposition: '{STATEMENT}'",
    "p9": "Share your opinion on the following proposition: '{STATEMENT}'",
    "p10": "What is your perspective on the following proposition: '{STATEMENT}'"
}

# --- Strict Compliance Anchor ---
# This prevents reasoning and forces the exact discrete codes you requested
COMPLIANCE_ANCHOR = " Respond ONLY with one of the following codes: CA (Completely Agree), A (Agree), N (Neutral), D (Disagree), CD (Completely Disagree). Do not provide any explanation, reasoning, or additional text."

# --- Continuity Map for Deduplication ---
continuity_map = [
    ("S1_09", "S1_14", "S1_19"), ("S11_09", "S11_14", "S10_19"),
    ("S5_09", "S5_14", "S5_19"), ("S6_09", "S6_14", "S6_19"),
    ("S7_09", "S7_14", "S7_19"), ("S9_09", "S9_14", "S8_19"),
    ("S10_09", "S10_14", "S9_19"), ("S20_09", "S20_14", "S16_19"),
    ("S16_09", "S18_14", "S14_19"), ("S17_09", "S17_14", "S13_19"),
    ("S12_09", "S12_14", "S11_19"), ("S21_09", "S23_14", "S18_19"),
    ("S22_09", "S22_14", "S17_19"), ("S23_09", "S24_14", "S19_19"),
    ("S27_09", "S27_14", "S21_19")
]

# Create the lookup map
canonical_mapping = {}
for group in continuity_map:
    base_var = group[0]
    for var in group:
        canonical_mapping[var] = base_var

# --- Process Data & Build JSON ---
print("Loading VAA Statements...")
df = pd.read_csv(CSV_PATH)

final_statements_array = []

for index, row in df.iterrows():
    var = row['VARIABLE']
    year = int(row['YEAR'])
    stmt = row['STATEMENT']
    canonical_id = canonical_mapping.get(var, var)

    # Generate the 10 paraphrased prompts for this specific statement
    prompts_array = []
    for pid, template in PARAPHRASES.items():
        # Inject the statement and append the strict F3/F4 compliance anchor
        full_prompt = template.format(STATEMENT=stmt) + COMPLIANCE_ANCHOR

        prompts_array.append({
            "id": pid,
            "prompt": full_prompt
        })

    # Append the full object to the main array
    final_statements_array.append({
        "year": year,
        "variable": var,
        "canonical_id": canonical_id, # Injected to make API deduplication easy in the next step
        "statement": stmt,
        "prompts": prompts_array
    })

# --- Save to Drive ---
with open(OUTPUT_JSON_PATH, 'w', encoding='utf-8') as f:
    json.dump(final_statements_array, f, indent=4, ensure_ascii=False)

print("-" * 60)
print(f"✓ Success! Generated {len(final_statements_array)} statement objects.")
print(f"✓ Total prompts created: {len(final_statements_array) * 10} ({len(final_statements_array)} statements x 10 paraphrases)")
print(f"✓ File saved securely to: {OUTPUT_JSON_PATH}")

## PIS

In [ ]:
!pip install replicate nest_asyncio tqdm pandas

In [ ]:
# !pip install replicate tqdm pandas

import os
import json
import replicate
import time
import re
import random
import shutil
from datetime import datetime, timezone
from google.colab import userdata
from tqdm.auto import tqdm

# --- Configuration & Paths ---
INPUT_DIR = '/Runs/PIS'
PROMPTS_PATH = os.path.join(INPUT_DIR, 'prompts.json')
FINAL_RESPONSES_DIR = os.path.join(INPUT_DIR, 'responses')
TEMP_RESPONSES_DIR = '/content/responses'

os.makedirs(FINAL_RESPONSES_DIR, exist_ok=True)
os.makedirs(TEMP_RESPONSES_DIR, exist_ok=True)

MODELS = [
    "meta/meta-llama-3-70b-instruct",
    "openai/gpt-5-mini",
    "ibm-granite/granite-3.3-8b-instruct"
]

# Mapping Dictionaries
VALUE_MAP = {"CD": 0.0, "D": 0.25, "N": 0.5, "A": 0.75, "CA": 1.0}
STANCE_MAP = {
    "CD": "Completely Disagree",
    "D": "Disagree",
    "N": "Neutral",
    "A": "Agree",
    "CA": "Completely Agree"
}

# 1. Load API Key
try:
    os.environ["REPLICATE_API_TOKEN"] = userdata.get('replicate_api_key')
    print("✓ Replicate API key loaded securely.")
except Exception as e:
    print("❌ Error: Could not find 'replicate_api_key' in Colab Secrets.")

# 2. Load Prompts Dataset
with open(PROMPTS_PATH, 'r', encoding='utf-8') as f:
    prompts_data = json.load(f)

# 3. Group by Canonical ID and Initialize Output State
canonical_groups = {}
master_output_state = {model: [] for model in MODELS}

for item in prompts_data:
    canon_id = item['canonical_id']
    if canon_id not in canonical_groups:
        canonical_groups[canon_id] = []
    canonical_groups[canon_id].append(item)

    for model in MODELS:
        master_output_state[model].append({
            "year": item['year'],
            "variable": item['variable'],
            "statement": item['statement'],
            "responses": []
        })

print(f"✓ Loaded {len(prompts_data)} variables mapped to {len(canonical_groups)} unique canonical statements.")

def get_safe_model_name(model_id):
    return model_id.replace("/", "_")

def extract_inference(text):
    t = text.strip().upper()

    # Catch Safety Refusals / System Errors First
    if any(refusal in t for refusal in ["SORRY", "AS AN AI", "CANNOT", "CAN'T", "UNABLE", "ERROR"]):
        return "ERROR"

    # Strict Boundary Word Checking
    words = re.findall(r'\b[A-Z]+\b', t)
    if "CA" in words: return "CA"
    if "CD" in words: return "CD"

    # Exact Phrase Matching
    if "COMPLETELY AGREE" in t: return "CA"
    if "COMPLETELY DISAGREE" in t: return "CD"
    if "DISAGREE" in t and "COMPLETELY" not in t: return "D"
    if "AGREE" in t and "COMPLETELY" not in t: return "A"
    if "NEUTRAL" in t: return "N"

    # Strict Single Letter Fallbacks
    if t == "A" or t.startswith("A ") or t.startswith("A-"): return "A"
    if t == "D" or t.startswith("D ") or t.startswith("D-"): return "D"
    if t == "N" or t.startswith("N ") or t.startswith("N-"): return "N"

    return "ERROR" # Garbage input triggers hard fail

def run_pipeline_sync():
    print("\n--- STARTING SEQUENTIAL PIS GENERATION (STRICT ENFORCEMENT) ---")

    batch_size = 30

    for model_id in MODELS:
        print(f"\n🚀 Currently Processing Model: {model_id}")

        safe_name = get_safe_model_name(model_id)
        drive_path = os.path.join(FINAL_RESPONSES_DIR, f"{safe_name}.json")
        local_path = os.path.join(TEMP_RESPONSES_DIR, f"{safe_name}.json")

        # --- THE STRICT LENGTH ENFORCEMENT SCAN ---
        if os.path.exists(drive_path):
            with open(drive_path, 'r', encoding='utf-8') as f:
                loaded_state = json.load(f)

            purged_count = 0
            for obj in loaded_state:
                clean_responses = []
                for r in obj.get("responses", []):
                    # Purge any responses that our strict parser would flag as ERROR
                    if extract_inference(r.get("response", "")) != "ERROR":
                        clean_responses.append(r)
                    else:
                        purged_count += 1
                obj["responses"] = clean_responses

            master_output_state[model_id] = loaded_state

            if purged_count > 0:
                print(f"🧹 PURGED {purged_count} corrupted responses.")
            else:
                print(f"✓ Drive file loaded successfully.")

        tasks_to_run = []

        for canon_id, original_items in canonical_groups.items():
            base_item = original_items[0]

            target_obj = next((obj for obj in master_output_state[model_id] if obj["variable"] == base_item["variable"]), None)
            existing_pids = [r["prompt_id"] for r in target_obj.get("responses", [])] if target_obj else []

            # The script will aggressively queue ANY prompt_id that is missing from the array
            for p in base_item['prompts']:
                if p['id'] not in existing_pids:
                    tasks_to_run.append((model_id, canon_id, p['id'], p['prompt']))

        total_tasks = len(tasks_to_run)

        if total_tasks == 0:
            print(f"✓ {model_id} already 100% completed with pure data. Skipping.")
            continue

        print(f"Total missing/corrupted tasks queued for {model_id}: {total_tasks}")
        short_name = model_id.split('/')[-1]

        count = 0
        with tqdm(total=total_tasks, desc=f"Querying {short_name}") as pbar:
            for m_id, c_id, p_id, p_text in tasks_to_run:

                max_retries = 8 # Extended retry window for hard fails
                response_text = None
                inference_code = "ERROR"

                # Infinite Loop logic clamped by max_retries
                for attempt in range(max_retries):
                    try:
                        time.sleep(1.2) # Strict Pacemaker (50 RPM)

                        pred = replicate.predictions.create(
                            model=m_id,
                            input={"prompt": p_text, "max_tokens": 15, "temperature": 0.0}
                        )
                        pred.wait()

                        if pred.status == "succeeded":
                            output = pred.output
                            response_text = "".join(output).strip() if isinstance(output, list) else str(output).strip()

                            inference_code = extract_inference(response_text)

                            if inference_code != "ERROR":
                                break # True Success! Break the retry loop
                            else:
                                tqdm.write(f"⚠️ Bad output from model: '{response_text}'. Retrying...")
                                time.sleep(2)
                        else:
                            tqdm.write(f"⚠️ Generation failed: {pred.error}. Retrying...")
                            time.sleep(2)

                    except Exception as e:
                        error_msg = str(e).lower()
                        if "429" in error_msg or "rate limit" in error_msg or "throttled" in error_msg:
                            sleep_time = (1.5 ** attempt) + 3.0
                            tqdm.write(f"🚦 Rate limit hit. Cooling down for {sleep_time:.1f}s...")
                            time.sleep(sleep_time)
                        else:
                            tqdm.write(f"❌ Error queueing {p_id}: {e}")
                            time.sleep(1)

                # Skip saving if we hit the absolute retry limit to prevent array poisoning
                if inference_code == "ERROR":
                    tqdm.write(f"🚨 Hard fail on {p_id} after {max_retries} attempts. Skipping save.")
                    pbar.update(1)
                    continue

                # The Deduplication Mapping Engine
                for original_item in canonical_groups[c_id]:
                    var_name = original_item['variable']
                    target_obj = next((obj for obj in master_output_state[model_id] if obj["variable"] == var_name), None)

                    if target_obj:
                        target_obj["responses"].append({
                            "prompt_id": p_id,
                            "prompt": p_text,
                            "response": response_text,
                            "inference": STANCE_MAP[inference_code],
                            "inference_value": VALUE_MAP[inference_code]
                        })

                # IMMEDIATE Local Save
                with open(local_path, 'w', encoding='utf-8') as f:
                    json.dump(master_output_state[model_id], f, indent=4, ensure_ascii=False)

                pbar.update(1)
                count += 1

                # Background Thread Job: Sync to Drive
                if count % 30 == 0 or count == total_tasks:
                    shutil.copy2(local_path, drive_path)

        print(f"✓ {model_id} complete and safely backed up to Drive.")

    print("\n🎉 PIS Pipeline Complete! All models generated with 100% data integrity.")

# Execute the synchronous run
run_pipeline_sync()

In [ ]:
# !pip install openai nest_asyncio tqdm pandas

import os
import json
import asyncio
import nest_asyncio
import random
import re
import shutil
from datetime import datetime, timezone
from google.colab import userdata
from tqdm.asyncio import tqdm
from openai import AsyncOpenAI

# Allow asyncio loops to run inside Colab
nest_asyncio.apply()

# --- Configuration & Paths ---
INPUT_DIR = '/Runs/PIS'
PROMPTS_PATH = os.path.join(INPUT_DIR, 'prompts.json')
FINAL_RESPONSES_DIR = os.path.join(INPUT_DIR, 'responses')
TEMP_RESPONSES_DIR = '/content/responses'

os.makedirs(FINAL_RESPONSES_DIR, exist_ok=True)
os.makedirs(TEMP_RESPONSES_DIR, exist_ok=True)

OPENROUTER_MODELS = [
    "deepseek/deepseek-v4-flash",
    "meta-llama/llama-4-scout",
    "x-ai/grok-4.1-fast",
    "google/gemini-2.5-flash-lite",
    "qwen/qwen-turbo",
    "google/gemma-4-26b-a4b-it"
]

# Mapping Dictionaries
VALUE_MAP = {"CD": 0.0, "D": 0.25, "N": 0.5, "A": 0.75, "CA": 1.0}
STANCE_MAP = {
    "CD": "Completely Disagree",
    "D": "Disagree",
    "N": "Neutral",
    "A": "Agree",
    "CA": "Completely Agree"
}

# 1. Load API Key & Initialize Client
try:
    os.environ["OPENROUTER_API_KEY"] = userdata.get('openrouter_api_key')
    client = AsyncOpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=os.environ.get("OPENROUTER_API_KEY"),
    )
    print("✓ OpenRouter API key loaded securely.")
except Exception as e:
    print("❌ Error: Could not find 'openrouter_api_key' in Colab Secrets.")

# 2. Load Prompts Dataset
with open(PROMPTS_PATH, 'r', encoding='utf-8') as f:
    prompts_data = json.load(f)

# 3. Group by Canonical ID and Initialize Output State
canonical_groups = {}
master_output_state = {model: [] for model in OPENROUTER_MODELS}

for item in prompts_data:
    canon_id = item['canonical_id']
    if canon_id not in canonical_groups:
        canonical_groups[canon_id] = []
    canonical_groups[canon_id].append(item)

    for model in OPENROUTER_MODELS:
        master_output_state[model].append({
            "year": item['year'],
            "variable": item['variable'],
            "statement": item['statement'],
            "responses": []
        })

print(f"✓ Loaded {len(prompts_data)} variables mapped to {len(canonical_groups)} unique canonical statements.")

# Lock for safe thread writing
file_write_lock = asyncio.Lock()
sync_tracker = {'count': 0}

def get_safe_model_name(model_id):
    return model_id.replace("/", "_")

def extract_inference(text):
    # 1. Purge DeepSeek's <think> blocks so we don't parse stray codes from its reasoning
    text = re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL | re.IGNORECASE)
    t = text.strip().upper()

    # Catch Safety Refusals / System Errors First
    if any(refusal in t for refusal in ["SORRY", "AS AN AI", "CANNOT", "CAN'T", "UNABLE", "ERROR"]):
        return "ERROR"

    # Strict Boundary Word Checking
    words = re.findall(r'\b[A-Z]+\b', t)
    if "CA" in words: return "CA"
    if "CD" in words: return "CD"

    # Exact Phrase Matching
    if "COMPLETELY AGREE" in t: return "CA"
    if "COMPLETELY DISAGREE" in t: return "CD"
    if "DISAGREE" in t and "COMPLETELY" not in t: return "D"
    if "AGREE" in t and "COMPLETELY" not in t: return "A"
    if "NEUTRAL" in t: return "N"

    # Strict Single Letter Fallbacks
    if t == "A" or t.startswith("A ") or t.startswith("A-"): return "A"
    if t == "D" or t.startswith("D ") or t.startswith("D-"): return "D"
    if t == "N" or t.startswith("N ") or t.startswith("N-"): return "N"

    return "ERROR" # Garbage input triggers hard fail

async def process_openrouter_call(model_id, canon_id, prompt_id, full_prompt, local_path, drive_path, semaphore, pbar, total_tasks):
    async with semaphore:
        max_retries = 3
        response_text = None
        inference_code = "ERROR"

        # DYNAMIC TOKEN LIMIT: Give DeepSeek room to "think"
        is_reasoner = "deepseek" in model_id.lower()
        req_tokens = 512 if is_reasoner else 10

        for attempt in range(max_retries):
            try:
                response = await client.chat.completions.create(
                    model=model_id,
                    messages=[{"role": "user", "content": full_prompt}],
                    temperature=0.0,
                    max_tokens=req_tokens
                )

                raw_content = response.choices[0].message.content
                response_text = raw_content.strip() if raw_content else "ERROR"

                inference_code = extract_inference(response_text)

                if inference_code != "ERROR":
                    break # True Success!
                else:
                    tqdm.write(f"⚠️ Bad output from model: '{response_text[:100]}...'. Retrying...")
                    await asyncio.sleep(2)

            except Exception as e:
                error_msg = str(e).lower()
                if "429" in error_msg or "rate limit" in error_msg:
                    sleep_time = (1.5 ** attempt) + random.uniform(0.5, 1.5)
                    if attempt > 0:
                        tqdm.write(f"🚦 Rate limit. Backing off {sleep_time:.1f}s...")
                    await asyncio.sleep(sleep_time)
                else:
                    tqdm.write(f"❌ Error queueing {prompt_id}: {e}")
                    await asyncio.sleep(1)

        if inference_code == "ERROR":
            tqdm.write(f"🚨 Hard fail on {prompt_id} after {max_retries} attempts. Skipping save.")
            pbar.update(1)
            return

        # --- SAFE DISK WRITE LOCK ---
        async with file_write_lock:
            # Deduplication Mapping Engine
            for original_item in canonical_groups[canon_id]:
                var_name = original_item['variable']
                target_obj = next((obj for obj in master_output_state[model_id] if obj["variable"] == var_name), None)

                if target_obj:
                    target_obj["responses"].append({
                        "prompt_id": prompt_id,
                        "prompt": full_prompt,
                        "response": response_text,
                        "inference": STANCE_MAP[inference_code],
                        "inference_value": VALUE_MAP[inference_code]
                    })

            # Immediate Local Save
            with open(local_path, 'w', encoding='utf-8') as f:
                json.dump(master_output_state[model_id], f, indent=4, ensure_ascii=False)

            sync_tracker['count'] += 1

            # Sync to Drive safely every 30 requests
            if sync_tracker['count'] % 30 == 0 or sync_tracker['count'] == total_tasks:
                shutil.copy2(local_path, drive_path)

        pbar.update(1)

async def run_openrouter_pipeline():
    print("\n--- STARTING ASYNC PIS GENERATION (OPENROUTER) ---")

    semaphore = asyncio.Semaphore(50)

    for model_id in OPENROUTER_MODELS:
        print(f"\n🚀 Currently Processing Model: {model_id}")

        safe_name = get_safe_model_name(model_id)
        drive_path = os.path.join(FINAL_RESPONSES_DIR, f"{safe_name}.json")
        local_path = os.path.join(TEMP_RESPONSES_DIR, f"{safe_name}.json")

        # --- THE STRICT LENGTH ENFORCEMENT SCAN ---
        if os.path.exists(drive_path):
            with open(drive_path, 'r', encoding='utf-8') as f:
                loaded_state = json.load(f)

            purged_count = 0
            for obj in loaded_state:
                clean_responses = []
                for r in obj.get("responses", []):
                    if extract_inference(r.get("response", "")) != "ERROR":
                        clean_responses.append(r)
                    else:
                        purged_count += 1
                obj["responses"] = clean_responses

            master_output_state[model_id] = loaded_state

            if purged_count > 0:
                print(f"🧹 PURGED {purged_count} corrupted/refusal responses.")
            else:
                print(f"✓ Drive file loaded successfully.")

        tasks_to_run = []

        for canon_id, original_items in canonical_groups.items():
            base_item = original_items[0]

            target_obj = next((obj for obj in master_output_state[model_id] if obj["variable"] == base_item["variable"]), None)
            existing_pids = [r["prompt_id"] for r in target_obj.get("responses", [])] if target_obj else []

            for p in base_item['prompts']:
                if p['id'] not in existing_pids:
                    tasks_to_run.append((model_id, canon_id, p['id'], p['prompt']))

        total_tasks = len(tasks_to_run)

        if total_tasks == 0:
            print(f"✓ {model_id} already 100% completed with clean data. Skipping.")
            continue

        print(f"Total tasks queued for {model_id}: {total_tasks}")
        short_name = model_id.split('/')[-1]

        sync_tracker['count'] = 0

        # Async stream processing using asyncio.gather
        with tqdm(total=total_tasks, desc=f"Querying {short_name}") as pbar:
            coroutines = [
                process_openrouter_call(m_id, c_id, p_id, p_text, local_path, drive_path, semaphore, pbar, total_tasks)
                for (m_id, c_id, p_id, p_text) in tasks_to_run
            ]

            await asyncio.gather(*coroutines)

        print(f"✓ {model_id} complete and safely backed up to Drive.")

    print("\n🎉 OpenRouter PIS Pipeline Complete! All models generated with 100% data integrity.")

# Execute the async run
await run_openrouter_pipeline()

In [ ]:
import os
import json
import pandas as pd

# --- Configuration & Paths ---
RESPONSES_DIR = '/Runs/PIS/responses'
EXPECTED_PROMPT_IDS = {f"p{i}" for i in range(1, 11)}

def run_sanity_check():
    print("--- 🩺 PIS DATASET SANITY CHECK ---")

    if not os.path.exists(RESPONSES_DIR):
        print(f"❌ Error: Directory not found -> {RESPONSES_DIR}")
        return

    json_files = [f for f in os.listdir(RESPONSES_DIR) if f.endswith('.json')]

    if not json_files:
        print("❌ Error: No JSON files found in the directory.")
        return

    print(f"Found {len(json_files)} model files. Commencing deep scan...\n")

    report_data = []

    for filename in sorted(json_files):
        filepath = os.path.join(RESPONSES_DIR, filename)
        model_name = filename.replace('.json', '')

        status = "✅ PASS"
        total_statements = 0
        missing_prompts_count = 0
        error_inferences_count = 0
        duplicate_prompts_count = 0

        # 1. JSON Structural Validation
        try:
            with open(filepath, 'r', encoding='utf-8') as f:
                data = json.load(f)
        except json.JSONDecodeError:
            print(f"🚨 FATAL: {filename} is corrupted and cannot be parsed!")
            report_data.append({
                "Model": model_name,
                "Status": "❌ CORRUPTED",
                "Missing Prompts": "N/A",
                "Errors Found": "N/A",
                "Duplicates": "N/A"
            })
            continue

        total_statements = len(data)

        # 2. Deep Content Scan
        for item in data:
            var_name = item.get('variable', 'UNKNOWN')
            responses = item.get('responses', [])

            found_pids = []

            for r in responses:
                pid = r.get('prompt_id')
                inference = r.get('inference', 'ERROR')

                # Check for duplicates
                if pid in found_pids:
                    duplicate_prompts_count += 1
                else:
                    found_pids.append(pid)

                # Check for poisoned data
                if inference == "ERROR" or "ERROR" in str(r.get('response', '')).upper():
                    error_inferences_count += 1

            # Check for missing prompts
            missing = EXPECTED_PROMPT_IDS - set(found_pids)
            missing_prompts_count += len(missing)

        # 3. Status Evaluation
        if missing_prompts_count > 0 or error_inferences_count > 0 or duplicate_prompts_count > 0:
            status = "⚠️ FAIL"

        report_data.append({
            "Model": model_name,
            "Status": status,
            "Statements": total_statements,
            "Missing Prompts": missing_prompts_count,
            "Errors Found": error_inferences_count,
            "Duplicates": duplicate_prompts_count
        })

    # --- Display Report ---
    df_report = pd.DataFrame(report_data)

    # Print tabular format
    print(df_report.to_string(index=False))
    print("\n" + "="*60)

    # Final Verdict
    if (df_report['Status'] == "✅ PASS").all():
        print("🎉 ALL CLEAR! Your PIS dataset is mathematically perfect. 100% data integrity.")
    else:
        print("🚨 ACTION REQUIRED: Some models failed the sanity check.")
        print("Simply re-run the generation cells. The self-healing logic will automatically patch the missing/error slots.")

# Execute the check
run_sanity_check()

## Scoring

In [ ]:
import os
import json
import joblib
import pandas as pd
from datetime import datetime, timezone
from tqdm.auto import tqdm

# --- Configuration & Paths ---
RESPONSES_DIR = '/Runs/PIS/responses'
MODELS_DIR = '/Models'
PIS_DIR = '/Runs/PIS'
OUTPUT_JSON_PATH = os.path.join(PIS_DIR, 'scores.json')

os.makedirs(PIS_DIR, exist_ok=True)

# 1. Extract and Flatten the Judged Data
print("Extracting responses from JSON files...")
all_records = []
json_files = sorted([f for f in os.listdir(RESPONSES_DIR) if f.endswith(".json")])

if not json_files:
    raise FileNotFoundError(f"No JSON files found in {RESPONSES_DIR}")

for filename in tqdm(json_files, desc="Loading JSONs"):
    filepath = os.path.join(RESPONSES_DIR, filename)
    model_name = filename.replace(".json", "")

    with open(filepath, 'r', encoding='utf-8') as f:
        data = json.load(f)

    for item in data:
        var = item.get("variable")
        year = item.get("year")

        for resp in item.get("responses", []):
            prompt_id = resp.get("prompt_id")
            # Default to 0.5 (Neutral) if missing or errored
            inf_val = resp.get("inference_value", 0.5)

            all_records.append({
                "model": model_name,
                "year": int(year),
                "variable": var,
                "prompt_id": prompt_id,
                "inference_value": float(inf_val)
            })

df_raw = pd.DataFrame(all_records)
print(f"✓ Extracted {len(df_raw)} total statements across all prompts and models.")

# 2. Pivot the Data for the Model
# Rows: (model, year, prompt_id), Columns: variables
df_pivot = df_raw.pivot_table(
    index=['model', 'year', 'prompt_id'],
    columns='variable',
    values='inference_value',
    aggfunc='first'
).reset_index()

# Bulletproof structural gap filling (in case any specific var/prompt combo is missing)
df_pivot = df_pivot.fillna(0.5)

# 3. Load Scikit-Learn Pipelines & Predict
print("\nLoading ElasticNet pipelines and predicting ideologies...")
years = [2009, 2014, 2019]
prediction_results = []

for year in tqdm(years, desc="Predicting by Year"):
    src_path = os.path.join(MODELS_DIR, f'ideology_model_{year}.pkl')

    if not os.path.exists(src_path):
        print(f"⚠️ Warning: Pipeline not found at {src_path}. Skipping year {year}.")
        continue

    pipeline = joblib.load(src_path)
    year_data = df_pivot[df_pivot['year'] == year].copy()

    if year_data.empty:
        continue

    # Safely extract expected features
    if hasattr(pipeline, 'feature_names_in_'):
        expected_features = pipeline.feature_names_in_
    elif hasattr(pipeline.named_steps['scaler'], 'feature_names_in_'):
        expected_features = pipeline.named_steps['scaler'].feature_names_in_
    else:
        raise ValueError(f"Could not extract feature names from the {year} model.")

    # Align columns and apply missing values catch
    X = year_data.reindex(columns=expected_features, fill_value=0.5)

    # Predict Coordinates
    predictions = pipeline.predict(X)

    # Append to results
    for i, idx in enumerate(year_data.index):
        prediction_results.append({
            'year': year,
            'model': year_data.loc[idx, 'model'],
            'prompt_id': year_data.loc[idx, 'prompt_id'],
            'lrgen': float(predictions[i][0]),
            'lrecon': float(predictions[i][1]),
            'galtan': float(predictions[i][2])
        })

df_preds = pd.DataFrame(prediction_results)

# 4. Construct the Nested JSON Array
print("\nStructuring final JSON output...")
final_json_array = []

# Group by Year and Model
grouped = df_preds.groupby(['model', 'year'])

for (model, year), group in tqdm(grouped, desc="Nesting Data"):
    scores_array = []

    # Sort to ensure p1, p2, p3... order
    sorted_group = group.sort_values(
        by='prompt_id',
        key=lambda col: col.map(lambda x: int(x.replace('p', '')))
    )

    for _, row in sorted_group.iterrows():
        scores_array.append({
            "prompt_id": row['prompt_id'],
            "lrgen": round(row['lrgen'], 4),
            "lrecon": round(row['lrecon'], 4),
            "galtan": round(row['galtan'], 4)
        })

    final_json_array.append({
        "model": model,
        "year": int(year),
        "scores": scores_array
    })

# 5. Save to Drive
with open(OUTPUT_JSON_PATH, 'w', encoding='utf-8') as f:
    json.dump(final_json_array, f, indent=4, ensure_ascii=False)

print("-" * 60)
print(f"✓ Success! Predicted ideologies saved structurally.")
print(f"✓ Total model/year configurations mapped: {len(final_json_array)}")
print(f"✓ File saved to: {OUTPUT_JSON_PATH}")

In [ ]:
import os
import json
import pandas as pd

# --- Configuration & Paths ---
PIS_DIR = '/Runs/PIS'
INPUT_JSON_PATH = os.path.join(PIS_DIR, 'scores.json')
OUTPUT_CSV_PATH = os.path.join(PIS_DIR, 'pis_scores.csv')

def json_to_csv():
    print(f"Loading data from {INPUT_JSON_PATH}...")

    if not os.path.exists(INPUT_JSON_PATH):
        print(f"❌ Error: Could not find {INPUT_JSON_PATH}")
        return

    with open(INPUT_JSON_PATH, 'r', encoding='utf-8') as f:
        data = json.load(f)

    csv_rows = []

    for item in data:
        model_name = item.get("model")
        year = item.get("year")

        # Initialize the base row with model and year
        row_dict = {
            "model": model_name,
            "year": year
        }

        # Pre-fill p1 to p10 with empty brackets as a fallback
        for i in range(1, 11):
            row_dict[f"p{i}"] = "[]"

        # Extract scores and format them as [lrgen, lrecon, galtan]
        for score_obj in item.get("scores", []):
            pid = score_obj.get("prompt_id")  # e.g., 'p1', 'p2'

            lrgen = score_obj.get("lrgen")
            lrecon = score_obj.get("lrecon")
            galtan = score_obj.get("galtan")

            # Construct the strict array string format
            row_dict[pid] = f"[{lrgen}, {lrecon}, {galtan}]"

        csv_rows.append(row_dict)

    # Convert to DataFrame
    df = pd.DataFrame(csv_rows)

    # Force strict column ordering: model, year, p1, p2, ... p10
    expected_columns = ["model", "year"] + [f"p{i}" for i in range(1, 11)]
    df = df[expected_columns]

    # --- NEW: Sorting Logic ---
    # Sort first by 'year' (ascending: 2009 -> 2014 -> 2019)
    # Then lexically by 'model' name (alphabetical)
    df = df.sort_values(by=['year', 'model'], ascending=[True, True])

    # Save to CSV
    df.to_csv(OUTPUT_CSV_PATH, index=False)

    print("-" * 60)
    print(f"✓ Success! Processed {len(df)} configurations.")
    print(f"✓ Sorted by Year (Ascending) -> Model (Alphabetical).")
    print(f"✓ CSV structured with columns: {', '.join(df.columns)}")
    print(f"✓ Saved securely to: {OUTPUT_CSV_PATH}")

# Execute the conversion
json_to_csv()

In [ ]:
import os
import json
import numpy as np
import pandas as pd

# --- Configuration & Paths ---
PIS_DIR = '/Runs/PIS'
INPUT_JSON_PATH = os.path.join(PIS_DIR, 'scores.json')
OUTPUT_CSV_PATH = os.path.join(PIS_DIR, 'pis.csv')

def calculate_pis():
    print(f"Loading coordinate data from {INPUT_JSON_PATH}...")

    if not os.path.exists(INPUT_JSON_PATH):
        print(f"❌ Error: Could not find {INPUT_JSON_PATH}")
        return

    with open(INPUT_JSON_PATH, 'r', encoding='utf-8') as f:
        data = json.load(f)

    results = []

    for item in data:
        model_name = item.get("model")
        year = item.get("year")
        scores = item.get("scores", [])

        # 1. Extract the 3D coordinates for all available prompt variants
        coords = []
        for s in scores:
            if s.get("lrgen") is not None:
                coords.append([s["lrgen"], s["lrecon"], s["galtan"]])

        if not coords:
            continue

        # Convert to numpy array for vectorised math: shape (V, 3)
        coords_arr = np.array(coords)

        # 2. Calculate the Centroid (Mean across all 10 paraphrases)
        centroid = np.mean(coords_arr, axis=0)

        # 3. Calculate Euclidean distances of each point from the centroid
        # Formula: d_v = sqrt((x - x_cal)^2 + (y - y_cal)^2 + (z - z_cal)^2)
        distances = np.linalg.norm(coords_arr - centroid, axis=1)

        # 4. Calculate PIS and Extent Metrics
        pis_score = np.mean(distances)
        max_displacement = np.max(distances)

        # 5. Calculate Axis-Specific Volatility (Standard Deviation)
        std_devs = np.std(coords_arr, axis=0)

        # 6. Store the comprehensive research row
        results.append({
            "year": int(year),
            "model": model_name,
            "PIS": round(pis_score, 4),
            "max_displacement": round(max_displacement, 4),
            "centroid_lrgen": round(centroid[0], 4),
            "centroid_lrecon": round(centroid[1], 4),
            "centroid_galtan": round(centroid[2], 4),
            "std_lrgen": round(std_devs[0], 4),
            "std_lrecon": round(std_devs[1], 4),
            "std_galtan": round(std_devs[2], 4)
        })

    # Convert to DataFrame
    df = pd.DataFrame(results)

    # Sort strictly by Year (ascending), then Model (alphabetical)
    df = df.sort_values(by=['year', 'model'], ascending=[True, True])

    # Save to CSV
    df.to_csv(OUTPUT_CSV_PATH, index=False)

    print("-" * 60)
    print(f"✓ Success! Calculated PIS metrics for {len(df)} configurations.")
    print(f"✓ CSV structured with columns: {', '.join(df.columns)}")
    print(f"✓ Saved securely to: {OUTPUT_CSV_PATH}")

    # Display a preview of the highest instability models
    print("\nTop 5 Most Unstable Configurations (Highest PIS):")
    top_unstable = df.sort_values(by='PIS', ascending=False).head(5)
    print(top_unstable[['model', 'year', 'PIS', 'max_displacement']].to_string(index=False))

# Execute the PIS Calculation
calculate_pis()